In [1]:
# compare_all_hybrids.py
"""
Comprehensive evaluation + comparison script for hybrid models.
Produces many publication-ready plots and CSVs for a paper.
Run: python compare_all_hybrids.py
"""

import os
import json
import pickle
import time
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score,
    roc_curve, auc, top_k_accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
from scipy.interpolate import interp1d

plt.rcParams["figure.dpi"] = 150
sns.set(style="whitegrid")

# ---------------- USER CONFIG ----------------
MODEL_PATHS = {
    "ALO+WOA": "hybrid_alo_woa_best_model.keras",
    "ALO+DA":  "hybrid_alo_da_best_model.keras",
    "ALO+PSO": "hybrid_alo_pso_best_model.keras",
    "PSO+WOA": "hybrid_pso_woa_best_model.keras",  # optional
}

TEST_DIR = "augmented_split_dataset/test"
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
RESCALE = True

OUT_ROOT = f"results/full_comparison_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_ROOT, exist_ok=True)
# ---------------------------------------------

def try_load_history(model_path):
    """Try several common history filenames (pkl/json/dict)."""
    p = Path(model_path)
    candidates = [
        p.with_suffix(p.suffix + ".history.pkl"),
        p.with_name(p.stem + "_history.pkl"),
        p.with_name(p.stem + ".history.json"),
        p.with_name(p.stem + "_history.json"),
        p.with_name(p.stem + ".history"),
        p.with_suffix(".history"),  # just in case
    ]
    for c in candidates:
        if c.exists():
            try:
                if c.suffix.endswith(".pkl"):
                    with open(c, "rb") as f:
                        data = pickle.load(f)
                        return data
                else:
                    with open(c, "r") as f:
                        return json.load(f)
            except Exception as e:
                print(f"[history] failed to load {c}: {e}")
    return None

def try_load_search_trace(model_prefix):
    """Check for search_trace JSONs produced during optimization, e.g. alo_woa_search_trace.json"""
    p = Path(model_prefix)
    candidates = [p.with_name(p.stem + "_search_trace.json"), p.with_name("search_trace.json")]
    for c in candidates:
        if c.exists():
            try:
                with open(c, "r") as f:
                    return json.load(f)
            except Exception as e:
                print(f"[search_trace] failed to load {c}: {e}")
    return None

def build_test_generator():
    datagen = ImageDataGenerator(rescale=1./255) if RESCALE else ImageDataGenerator()
    gen = datagen.flow_from_directory(
        directory=TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    return gen

# ---- plotting helpers ----
def save_fig(fig, out_path):
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

def plot_acc_loss_from_history(history, out_path_prefix, model_label):
    if history is None:
        return None
    hist = history.history if hasattr(history, "history") else history
    epochs = range(1, len(next(iter(hist.values()))) + 1)
    fig, axes = plt.subplots(1,2, figsize=(10,4))
    # accuracy
    acc_key = next((k for k in ("accuracy","acc") if k in hist), None)
    val_acc_key = next((k for k in ("val_accuracy","val_acc") if k in hist), None)
    if acc_key or val_acc_key:
        if acc_key: axes[0].plot(epochs, hist[acc_key], label="train")
        if val_acc_key: axes[0].plot(epochs, hist[val_acc_key], label="val")
        axes[0].set_title(f"Accuracy ({model_label})"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend(); axes[0].grid(True)
    # loss
    if "loss" in hist:
        axes[1].plot(epochs, hist["loss"], label="train")
    if "val_loss" in hist:
        axes[1].plot(epochs, hist["val_loss"], label="val")
    axes[1].set_title(f"Loss ({model_label})"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend(); axes[1].grid(True)
    save_fig(fig, out_path_prefix + "_acc_loss.png")
    return hist

def plot_confusion_matrices(cm, class_names, out_prefix, title_prefix=""):
    # raw and normalized
    cm_raw = cm
    cm_norm = (cm_raw.astype('float') / np.maximum(cm_raw.sum(axis=1)[:, None], 1))
    for mat, suffix, t in [(cm_raw, "raw", "Confusion (raw)"), (cm_norm, "norm", "Confusion (normalized)")]:
        fig, ax = plt.subplots(1,1,figsize=(6,6))
        sns.heatmap(mat, annot=True, fmt=".2f" if suffix=="norm" else "d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{title_prefix} {t}")
        plt.xticks(rotation=90)
        save_fig(fig, f"{out_prefix}_confmat_{suffix}.png")

def plot_pr_roc(y_true, y_prob, class_names, out_prefix, model_label):
    n_classes = y_prob.shape[1]
    Y = label_binarize(y_true, classes=np.arange(n_classes))
    # PR
    precision = {}; recall = {}; ap = {}
    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(Y[:, i], y_prob[:, i])
        ap[i] = average_precision_score(Y[:, i], y_prob[:, i])
    precision["micro"], recall["micro"], _ = precision_recall_curve(Y.ravel(), y_prob.ravel())
    ap["micro"] = average_precision_score(Y, y_prob, average="micro")
    # PR plot
    fig, ax = plt.subplots(figsize=(8,6))
    ax.step(recall["micro"], precision["micro"], where='post', label=f"micro (AP={ap['micro']:.3f})", lw=2)
    colors = cycle(plt.rcParams['axes.prop_cycle'].by_key()['color'])
    for i,color in zip(range(n_classes), colors):
        ax.step(recall[i], precision[i], where='post', label=f"{class_names[i]} (AP={ap[i]:.3f})", alpha=0.8)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"Precision-Recall ({model_label})")
    ax.legend(loc="best", fontsize="small"); ax.grid(True)
    save_fig(fig, f"{out_prefix}_pr.png")

    # ROC
    fpr = {}; tpr = {}; roc_auc = {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(Y[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(Y.ravel(), y_prob.ravel()); roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    # macro
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr; tpr["macro"] = mean_tpr; roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    fig, ax = plt.subplots(figsize=(8,6))
    ax.plot(fpr["micro"], tpr["micro"], label=f"micro (AUC={roc_auc['micro']:.3f})", lw=2)
    ax.plot(fpr["macro"], tpr["macro"], label=f"macro (AUC={roc_auc['macro']:.3f})", lw=2, linestyle='--')
    colors = cycle(plt.rcParams['axes.prop_cycle'].by_key()['color'])
    for i,color in zip(range(n_classes), colors):
        ax.plot(fpr[i], tpr[i], lw=1, label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})", alpha=0.8)
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlim([-0.02,1.02]); ax.set_ylim([-0.02,1.02])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_title(f"ROC ({model_label})")
    ax.legend(loc='lower right', fontsize='small'); ax.grid(True)
    save_fig(fig, f"{out_prefix}_roc.png")
    return {"ap": ap, "auc": roc_auc}

def plot_calibration_curve(y_true, y_prob, out_path, model_label, n_bins=10):
    confidences = np.max(y_prob, axis=1)
    preds = np.argmax(y_prob, axis=1)
    correct = (preds == y_true).astype(int)
    prob_true, prob_pred = calibration_curve(correct, confidences, n_bins=n_bins)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(prob_pred, prob_true, marker='o', label='Model')
    ax.plot([0,1],[0,1],'k--', label='Perfect')
    ax.set_xlabel('Predicted probability'); ax.set_ylabel('Fraction of positives')
    ax.set_title(f'Calibration ({model_label})'); ax.legend(); ax.grid(True)
    save_fig(fig, out_path)

def save_misclassified_examples(test_gen, y_true, y_pred, out_dir, max_examples=36):
    mis_idx = np.where(y_true != y_pred)[0]
    if mis_idx.size == 0: return
    sel = mis_idx[:max_examples]
    # generator exposes filepaths
    filepaths = getattr(test_gen, 'filepaths', None)
    filenames = getattr(test_gen, 'filenames', None)
    src_paths = None
    if filepaths is not None:
        src_paths = filepaths
    elif filenames is not None and hasattr(test_gen, 'directory'):
        src_paths = [os.path.join(test_gen.directory, f) for f in filenames]
    out_misdir = os.path.join(out_dir, "misclassified")
    os.makedirs(out_misdir, exist_ok=True)
    rows = []
    for i, idx in enumerate(sel):
        if src_paths is not None:
            src = src_paths[idx]
            basename = os.path.basename(src)
            dst = os.path.join(out_misdir, f"{i:03d}_idx{idx}_t{y_true[idx]}_p{y_pred[idx]}_{basename}")
            try:
                from shutil import copyfile
                copyfile(src, dst)
            except Exception:
                pass
            rows.append({"idx": idx, "true": int(y_true[idx]), "pred": int(y_pred[idx]), "file": dst})
    pd.DataFrame(rows).to_csv(os.path.join(out_misdir, "misclassified_list.csv"), index=False)

# ---- evaluate single model ----
def evaluate_model_full(model_label, model_path, test_gen, out_root):
    print(f"\n--- Evaluating {model_label} ({model_path}) ---")
    model_out = os.path.join(out_root, model_label.replace(" ", "_"))
    os.makedirs(model_out, exist_ok=True)

    model = load_model(model_path)
    hist = try_load_history(model_path)
    if hist is not None:
        _ = plot_acc_loss_from_history(hist, os.path.join(model_out, model_label.replace(" ", "_")), model_label)
        # save raw history for later cross-model plots
        with open(os.path.join(model_out, "history.obj"), "wb") as f:
            pickle.dump(hist, f)

    # predict
    y_prob = model.predict(test_gen, verbose=1)
    if y_prob.ndim == 1 or (y_prob.ndim == 2 and y_prob.shape[1] == 1):
        p = y_prob.ravel()
        y_prob = np.vstack([1-p, p]).T
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    # classification report
    crep = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=False)
    with open(os.path.join(model_out, "classification_report.txt"), "w") as f:
        f.write(crep)
    # save raw preds
    np.savez_compressed(os.path.join(model_out, "preds.npz"), y_true=y_true, y_pred=y_pred, y_prob=y_prob, class_names=class_names)

    # confusion
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrices(cm, class_names, os.path.join(model_out, "confusion"), title_prefix=model_label)

    # per-class metrics
    f1s = f1_score(y_true, y_pred, average=None)
    precisions = precision_score(y_true, y_pred, average=None, zero_division=0)
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    df_pc = pd.DataFrame({"class": class_names, "precision": precisions, "recall": recalls, "f1": f1s})
    df_pc.to_csv(os.path.join(model_out, "per_class_metrics.csv"), index=False)

    # PR/ROC
    try:
        prroc_stats = plot_pr_roc(y_true, y_prob, class_names, model_out, model_label)
    except Exception as e:
        print("PR/ROC error:", e); prroc_stats = None

    # calibration
    try:
        plot_calibration_curve(y_true, y_prob, os.path.join(model_out, "calibration.png"), model_label)
    except Exception as e:
        print("Calibration error:", e)

    # top-k
    try:
        top1 = top_k_accuracy_score(y_true, y_prob, k=1)
        top3 = top_k_accuracy_score(y_true, y_prob, k=3) if y_prob.shape[1] >= 3 else None
    except Exception as e:
        top1, top3 = None, None
    # summary
    summary = {
        "model": model_label,
        "accuracy": float((y_pred == y_true).mean()),
        "macro_f1": float(np.mean(f1s)),
        "top1": float(top1) if top1 is not None else None,
        "top3": float(top3) if top3 is not None else None,
        "num_classes": len(class_names)
    }
    with open(os.path.join(model_out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # misclassified
    save_misclassified_examples(test_gen, y_true, y_pred, model_out, max_examples=36)
    return summary, df_pc, prroc_stats

# ---- cross-model plotting ----
def align_and_plot_val_accuracy(model_histories, out_path, labels):
    """
    Align val_accuracy curves of different lengths by interpolation and plot overlay.
    model_histories: dict label -> history dict (with val_accuracy key)
    """
    # collect val_accuracy arrays
    val_accs = {}
    max_epochs = 0
    for lbl, hist in model_histories.items():
        if hist is None: continue
        h = hist.history if hasattr(hist, "history") else hist
        key = next((k for k in ("val_accuracy","val_acc") if k in h), None)
        if key is None: continue
        arr = np.array(h[key])
        val_accs[lbl] = arr
        max_epochs = max(max_epochs, arr.shape[0])
    if not val_accs:
        return
    # create common epoch axis and interpolate
    epoch_axis = np.arange(1, max_epochs+1)
    plt.figure(figsize=(8,5))
    for lbl, arr in val_accs.items():
        ep = np.arange(1, len(arr)+1)
        f = interp1d(ep, arr, kind='linear', bounds_error=False, fill_value=(arr[0], arr[-1]))
        plt.plot(epoch_axis, f(epoch_axis), label=lbl)
    plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy"); plt.title("Validation Accuracy Comparison"); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.savefig(out_path); plt.close()

def plot_summary_bars(summaries, out_dir):
    df = pd.DataFrame(summaries).set_index("model")
    df[['accuracy','macro_f1']].plot(kind='bar', figsize=(8,4))
    plt.title("Model comparison: Accuracy & Macro-F1"); plt.ylim(0,1); plt.grid(axis='y'); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "summary_acc_macrof1.png")); plt.close()
    if 'top1' in df.columns:
        df['top1'].plot(kind='bar', figsize=(6,3)); plt.title("Top-1 accuracy"); plt.ylim(0,1); plt.tight_layout(); plt.savefig(os.path.join(out_dir, "summary_top1.png")); plt.close()

def plot_per_class_heatmap(per_class_dfs, out_path):
    # per_class_dfs: dict model-> DataFrame with columns class,f1
    combined = None
    for model, df in per_class_dfs.items():
        s = df.set_index('class')['f1'].rename(model)
        combined = s.to_frame() if combined is None else combined.join(s, how='outer')
    if combined is None: return
    combined = combined.fillna(0)
    fig, ax = plt.subplots(figsize=(max(6, 0.6*combined.shape[0]), max(4, 0.4*combined.shape[1])))
    sns.heatmap(combined.T, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_xlabel("Class"); ax.set_ylabel("Model"); ax.set_title("Per-class F1 (models vs classes)")
    plt.tight_layout(); plt.savefig(out_path); plt.close()

def plot_search_traces(search_traces, out_path):
    # search_traces: dict model -> trace object with generation fitness arrays
    plt.figure(figsize=(8,4))
    for model, trace in search_traces.items():
        # expect trace like {'gens': [{'gen':0,'fitness':[...]} , ...]} or {'fitness': [best_so_far_per_gen]}
        if isinstance(trace, dict) and 'generations' in trace:
            bests = [g.get('best', None) for g in trace['generations']]
            plt.plot(range(1, len(bests)+1), bests, label=model)
        elif isinstance(trace, list):
            # maybe list of gens where each has fitness array
            bests = []
            for g in trace:
                if isinstance(g, dict) and 'fitness' in g:
                    bests.append(np.max(g['fitness']))
                else:
                    # try simple numeric list
                    try:
                        bests.append(float(g))
                    except Exception:
                        pass
            if bests:
                plt.plot(range(1, len(bests)+1), bests, label=model)
        elif isinstance(trace, dict) and 'best_per_gen' in trace:
            plt.plot(range(1, len(trace['best_per_gen'])+1), trace['best_per_gen'], label=model)
    plt.xlabel("Generation"); plt.ylabel("Best validation accuracy"); plt.title("Search convergence (best per generation)"); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.savefig(out_path); plt.close()

# ---- main ----
def main():
    test_gen = build_test_generator()

    summaries = []
    per_class_dfs = {}
    histories = {}
    search_traces = {}

    for label, path in MODEL_PATHS.items():
        if not os.path.exists(path):
            print(f"[WARN] Model not found: {path} (skipping {label})")
            continue
        s, df_pc, prroc = evaluate_model_full(label, path, test_gen, OUT_ROOT)
        summaries.append(s)
        per_class_dfs[label] = df_pc
        # load history if saved earlier
        hist = try_load_history(path)
        histories[label] = hist
        # try search trace named after the model prefix
        st = try_load_search_trace(Path(path).stem)
        if st is not None:
            search_traces[label] = st

    # Save combined summary CSV
    if summaries:
        pd.DataFrame(summaries).to_csv(os.path.join(OUT_ROOT, "models_summary.csv"), index=False)
        # cross-model plots
        plot_summary_bars(summaries, OUT_ROOT)
        # overlay val-accuracy (if histories exist)
        align_and_plot_val_accuracy(histories, os.path.join(OUT_ROOT, "val_accuracy_comparison.png"), list(histories.keys()))
        # per-class heatmap
        plot_per_class_heatmap(per_class_dfs, os.path.join(OUT_ROOT, "per_class_f1_heatmap.png"))
        # search traces if any
        if search_traces:
            plot_search_traces(search_traces, os.path.join(OUT_ROOT, "search_convergence.png"))

    print(f"\nAll outputs saved to: {OUT_ROOT}")

if __name__ == "__main__":
    main()


Found 4602 images belonging to 10 classes.

--- Evaluating ALO+WOA (hybrid_alo_woa_best_model.keras) ---


2025-09-13 16:30:42.640104: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2025-09-13 16:30:42.640146: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-09-13 16:30:42.640149: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-09-13 16:30:42.640358: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-13 16:30:42.640370: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/simarkalsi/Projects/leafClassification/leaf/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class shoul

144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step

--- Evaluating ALO+DA (hybrid_alo_da_best_model.keras) ---
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step

--- Evaluating ALO+PSO (hybrid_alo_pso_best_model.keras) ---
144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step

--- Evaluating PSO+WOA (hybrid_pso_woa_best_model.keras) ---
144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step

All outputs saved to: results/full_comparison_20250913_163042


/var/folders/c0/fkwzzmb92yldm18l2b0gx5wh0000gn/T/ipykernel_13545/3482803961.py:351: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout(); plt.savefig(out_path); plt.close()


In [15]:
"""
generate_big_confusion_matrices.py

What it does:
- Loads each .keras model listed in MODEL_PATHS
- Builds test generator from augmented_split_dataset/test (rescale=1./255, shuffle=False)
- Predicts on the test set (batch-wise)
- Creates and saves BIG-CELL confusion matrices (PNG + PDF + CSV) into results/confusion_matrices/<MODEL>/

Adjust cell_size to make cells bigger/smaller (per-class cell width/height in inches).
"""

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix

# ----------------- USER CONFIG -----------------
MODEL_PATHS = {
    "ALO_WOA": "hybrid_alo_woa_best_model.keras",
    "ALO_DA" : "hybrid_alo_da_best_model.keras",
    "ALO_PSO": "hybrid_alo_pso_best_model.keras",
    "PSO_WOA": "hybrid_pso_woa_best_model.keras",  # optional; will be skipped if missing
}

TEST_DIR = "augmented_split_dataset/test"
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
RESCALE = True

OUT_ROOT = "results/confusion_matrices_bigcells"
os.makedirs(OUT_ROOT, exist_ok=True)

# Size of each square cell in inches. Increase to make cells bigger.
# Example: for 10 classes and cell_size=1.5, figure will be 15 x 15 inches.
CELL_SIZE = 1.5

# Font sizes for labels/titles and numbers inside cells.
LABEL_FONTSIZE = 12     # keeps axis labels and title readable but not huge
ANNOT_FONTSIZE = 12     # font size for numbers inside cells (kept same; cells get bigger)
DPI = 300
# ------------------------------------------------

def build_test_generator():
    datagen = ImageDataGenerator(rescale=1./255) if RESCALE else ImageDataGenerator()
    gen = datagen.flow_from_directory(
        directory=TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    return gen

def plot_confusion_matrix_bigcells(cm,
                                   class_names,
                                   out_path,
                                   normalize=True,
                                   cell_size=CELL_SIZE,
                                   label_fontsize=LABEL_FONTSIZE,
                                   annot_fontsize=ANNOT_FONTSIZE,
                                   dpi=DPI,
                                   cmap="Blues"):
    """
    cm: square confusion matrix (counts)
    class_names: list of class labels (strings)
    out_path: file path to save (png/pdf/svg). Use full path including filename.
    normalize: if True, display row-normalized percentages; otherwise raw counts
    cell_size: width/height in inches for each class cell (float)
    """
    import numpy as _np
    import matplotlib.pyplot as _plt
    import seaborn as _sns

    cm = _np.asarray(cm)
    if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
        raise ValueError("cm must be square (n_classes x n_classes)")

    n = cm.shape[0]
    # Figure size scales with number of classes * cell_size
    fig_w = max(6, n * cell_size)
    fig_h = max(4, n * cell_size)

    _plt.rcParams.update({'font.size': label_fontsize})
    fig, ax = _plt.subplots(figsize=(fig_w, fig_h))

    # Compute displayed values
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True).astype(float)
        with _np.errstate(divide='ignore', invalid='ignore'):
            data = (cm / _np.maximum(row_sums, 1)) * 100.0  # percent per row
        fmt = ".1f"
        annot_kws = {"size": annot_fontsize}
    else:
        data = cm
        fmt = "d"
        annot_kws = {"size": annot_fontsize}

    # Use seaborn heatmap to draw squares sized by figure dimensions
    _sns.set(style="white")
    im = _sns.heatmap(data,
                      annot=True,
                      fmt=fmt,
                      cmap=cmap,
                      cbar=True,
                      xticklabels=class_names,
                      yticklabels=class_names,
                      linewidths=0.5,
                      linecolor="gray",
                      ax=ax,
                      annot_kws=annot_kws,
                      square=True)

    # Keep label fonts controlled (not scaling with figure)
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=label_fontsize)
    ax.set_yticklabels(class_names, rotation=0, fontsize=label_fontsize)
    ax.set_xlabel("Predicted label", fontsize=label_fontsize + 0)
    ax.set_ylabel("True label", fontsize=label_fontsize + 0)
    ax.set_title("Confusion Matrix" + (" (row-normalized %)" if normalize else " (counts)"), fontsize=label_fontsize + 2)

    # Tweak colorbar tick size
    cbar = im.collections[0].colorbar
    if cbar is not None:
        cbar.ax.tick_params(labelsize=max(8, label_fontsize - 1))

    _plt.tight_layout()

    # Ensure directory exists
    out_dir = os.path.dirname(out_path)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    # Save (filename extension controls PNG vs PDF/SVG)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    _plt.close(fig)
    print(f"Saved: {out_path}")

def process_model(label, model_path, test_gen):
    print(f"\n--- Processing {label} ---")
    model = load_model(model_path)
    print("Model loaded:", model_path)

    # Predict (batchwise)
    print("Predicting on test set...")
    y_prob = model.predict(test_gen, verbose=1)
    if y_prob.ndim == 1 or (y_prob.ndim == 2 and y_prob.shape[1] == 1):
        p = y_prob.ravel()
        y_prob = np.vstack([1-p, p]).T
    y_pred = np.argmax(y_prob, axis=1)

    if not hasattr(test_gen, 'classes'):
        raise RuntimeError("test_gen must expose .classes (use flow_from_directory with shuffle=False).")
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    # Confusion matrix counts
    cm = confusion_matrix(y_true, y_pred)

    model_out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(model_out_dir, exist_ok=True)

    # Save CSV of raw counts
    csv_path = os.path.join(model_out_dir, f"confusion_matrix_{label}.csv")
    np.savetxt(csv_path, cm, fmt="%d", delimiter=",", header=",".join(class_names), comments="")
    print("Saved CSV:", csv_path)

    # Save big-cell normalized (row %) PNG and PDF
    png_norm = os.path.join(model_out_dir, f"confusion_matrix_{label}_bigcells_norm.png")
    pdf_norm = os.path.join(model_out_dir, f"confusion_matrix_{label}_bigcells_norm.pdf")
    plot_confusion_matrix_bigcells(cm, class_names, png_norm, normalize=True,
                                   cell_size=CELL_SIZE, label_fontsize=LABEL_FONTSIZE,
                                   annot_fontsize=ANNOT_FONTSIZE, dpi=DPI)
    plot_confusion_matrix_bigcells(cm, class_names, pdf_norm, normalize=True,
                                   cell_size=CELL_SIZE, label_fontsize=LABEL_FONTSIZE,
                                   annot_fontsize=ANNOT_FONTSIZE, dpi=DPI)

    # Save big-cell counts version too
    png_counts = os.path.join(model_out_dir, f"confusion_matrix_{label}_bigcells_counts.png")
    plot_confusion_matrix_bigcells(cm, class_names, png_counts, normalize=False,
                                   cell_size=CELL_SIZE, label_fontsize=LABEL_FONTSIZE,
                                   annot_fontsize=ANNOT_FONTSIZE, dpi=DPI)

    print(f"Finished {label}. Outputs in: {model_out_dir}")

def main():
    test_gen = build_test_generator()

    for label, path in MODEL_PATHS.items():
        if not os.path.exists(path):
            print(f"[WARN] Model not found: {path} — skipping {label}")
            continue
        try:
            process_model(label, path, test_gen)
        except Exception as e:
            print(f"[ERROR] {label} failed: {e}")

if __name__ == "__main__":
    main()


Found 4602 images belonging to 10 classes.

--- Processing ALO_WOA ---
Model loaded: hybrid_alo_woa_best_model.keras
Predicting on test set...
  1/144 ━━━━━━━━━━━━━━━━━━━━ 19s 138ms/step

/Users/simarkalsi/Projects/leafClassification/leaf/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step
Saved CSV: results/confusion_matrices_bigcells/ALO_WOA/confusion_matrix_ALO_WOA.csv
Saved: results/confusion_matrices_bigcells/ALO_WOA/confusion_matrix_ALO_WOA_bigcells_norm.png
Saved: results/confusion_matrices_bigcells/ALO_WOA/confusion_matrix_ALO_WOA_bigcells_norm.pdf
Saved: results/confusion_matrices_bigcells/ALO_WOA/confusion_matrix_ALO_WOA_bigcells_counts.png
Finished ALO_WOA. Outputs in: results/confusion_matrices_bigcells/ALO_WOA

--- Processing ALO_DA ---
Model loaded: hybrid_alo_da_best_model.keras
Predicting on test set...
144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step
Saved CSV: results/confusion_matrices_bigcells/ALO_DA/confusion_matrix_ALO_DA.csv
Saved: results/confusion_matrices_bigcells/ALO_DA/confusion_matrix_ALO_DA_bigcells_norm.png
Saved: results/confusion_matrices_bigcells/ALO_DA/confusion_matrix_ALO_DA_bigcells_norm.pdf
Saved: results/confusion_matrices_bigcells/ALO_DA/confusion_matrix_ALO_DA_bigcells_counts.png
Finished ALO_DA.

In [31]:
# generate_metrics_heatmaps_metricsY.py
"""
Generate big-cell heatmaps (metrics on Y-axis, classes on X-axis) for each hybrid model.
Saves PNG and PDF per model and CSV with numeric values.
"""

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import precision_recall_fscore_support

# ---------------- USER CONFIG ----------------
MODEL_PATHS = {
    "ALO_WOA": "hybrid_alo_woa_best_model.keras",
    "ALO_DA" : "hybrid_alo_da_best_model.keras",
    "ALO_PSO": "hybrid_alo_pso_best_model.keras",
    "PSO_WOA": "hybrid_pso_woa_best_model.keras",  # optional; remove or update if you don't have it
}

TEST_DIR = "augmented_split_dataset/test"
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
RESCALE = True

OUT_ROOT = "results/metrics_heatmaps_metricsY"
os.makedirs(OUT_ROOT, exist_ok=True)

# Big-cell sizing: inches per cell (increase to make each cell larger)
CELL_SIZE = 1.6

# Fonts and DPI
LABEL_FONTSIZE = 12   # axis labels and ticks
ANNOT_FONTSIZE = 12   # annotation inside cells
DPI = 300
# ---------------------------------------------

def build_test_generator():
    """Builds flow_from_directory generator for test set (shuffle=False)."""
    datagen = ImageDataGenerator(rescale=1./255) if RESCALE else ImageDataGenerator()
    gen = datagen.flow_from_directory(
        directory=TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    return gen

def plot_bigcell_heatmap_metricsY(data,
                                  metric_labels,
                                  class_labels,
                                  out_path,
                                  cell_size=CELL_SIZE,
                                  annot=True,
                                  annot_fontsize=ANNOT_FONTSIZE,
                                  label_fontsize=LABEL_FONTSIZE,
                                  cmap="YlGnBu",
                                  fmt=".3f",
                                  dpi=DPI,
                                  title="Per-class metrics (metrics on Y-axis)"):
    """
    Plot big-cell heatmap with metrics on Y-axis and classes on X-axis.
      - data: 2D array with shape (n_metrics, n_classes)
      - metric_labels: list of row names, e.g. ["Precision","Recall","F1"]
      - class_labels: list of column names (class names)
    """
    import numpy as _np
    import matplotlib.pyplot as _plt
    import seaborn as _sns

    data = _np.asarray(data)
    if data.ndim != 2:
        raise ValueError("data must be 2D (n_metrics x n_classes)")

    n_rows, n_cols = data.shape

    fig_w = max(6, n_cols * cell_size)
    fig_h = max(3, n_rows * cell_size)

    fig, ax = _plt.subplots(figsize=(fig_w, fig_h))
    _sns.set(style="white")

    # Draw heatmap: big cells, annotated
    _sns.heatmap(data,
                 annot=annot,
                 fmt=fmt,
                 cmap=cmap,
                 xticklabels=class_labels,
                 yticklabels=metric_labels,
                 linewidths=0.6,
                 linecolor="gray",
                 annot_kws={"size": annot_fontsize, "weight": "bold"},
                 cbar=True,
                 ax=ax)

    # Keep label fonts controlled
    ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=label_fontsize)
    ax.set_yticklabels(metric_labels, rotation=0, fontsize=label_fontsize)
    ax.set_title(title, fontsize=label_fontsize + 2)

    # Adjust colorbar tick label sizes (if present)
    # (sns heatmap stores colorbar on the mappable)
    try:
        cbar = ax.collections[0].colorbar
        if cbar is not None:
            cbar.ax.tick_params(labelsize=max(8, label_fontsize - 1))
    except Exception:
        pass

    _plt.tight_layout()
    out_dir = os.path.dirname(out_path)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    _plt.close(fig)
    print(f"Saved heatmap: {out_path}")

def process_model(label, model_path, test_gen):
    """Load model, predict, compute per-class metrics and save heatmaps/CSV."""
    print(f"\n--- Processing model: {label} ---")
    model = load_model(model_path)
    print("Loaded:", model_path)

    # Predict on test set (batch-wise)
    print("Predicting on test set...")
    y_prob = model.predict(test_gen, verbose=1)
    # If binary single-column probs → convert to two-column proba
    if y_prob.ndim == 1 or (y_prob.ndim == 2 and y_prob.shape[1] == 1):
        p = y_prob.ravel()
        y_prob = np.vstack([1 - p, p]).T

    y_pred = np.argmax(y_prob, axis=1)

    if not hasattr(test_gen, "classes"):
        raise RuntimeError("test_gen must expose .classes (use flow_from_directory with shuffle=False).")
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())
    n_classes = len(class_names)

    # Compute per-class precision, recall, f1, supports
    precisions, recalls, f1s, supports = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_classes)), zero_division=0
    )

    # Build matrix with shape (3, n_classes) → rows: Precision, Recall, F1
    matrix = np.vstack([precisions, recalls, f1s])  # values in [0,1]

    # Save numeric CSV
    model_out = os.path.join(OUT_ROOT, label)
    os.makedirs(model_out, exist_ok=True)
    csv_path = os.path.join(model_out, f"per_class_metrics_{label}.csv")
    with open(csv_path, "w") as f:
        f.write("class,precision,recall,f1,support\n")
        for cname, p, r, f1v, s in zip(class_names, precisions, recalls, f1s, supports):
            f.write(f"{cname},{p:.6f},{r:.6f},{f1v:.6f},{int(s)}\n")
    print("Saved CSV:", csv_path)

    # Plot heatmap with metrics on Y axis (rows), classes on X axis (cols)
    row_labels = ["Precision", "Recall", "F1"]
    png_path = os.path.join(model_out, f"per_class_metrics_heatmap_{label}_metricsY.png")
    pdf_path = os.path.join(model_out, f"per_class_metrics_heatmap_{label}_metricsY.pdf")

    # We show values as decimals (0..1). Use fmt ".3f"
    plot_bigcell_heatmap_metricsY = plot_bigcell_heatmap_metricsY = None
    # define local wrapper from above function name to avoid reuse confusion
    def _plotter(data, metrics, classes, outp):
        plot_bigcell_heatmap_metricsY(
            data,
            metric_labels=metrics,
            class_labels=classes,
            out_path=outp,
            cell_size=CELL_SIZE,
            annot=True,
            annot_fontsize=ANNOT_FONTSIZE,
            label_fontsize=LABEL_FONTSIZE,
            cmap="YlGnBu",
            fmt=".3f",
            dpi=DPI,
            title=f"{label} - Per-class Metrics (Precision, Recall, F1)"
        )

    # call the provided plotting function (we'll actually call the function defined below)
    # Save PNG + PDF
    plot_bigcell_heatmap_metricsY = globals().get("plot_bigcell_heatmap_metricsY")
    if plot_bigcell_heatmap_metricsY is None:
        # fallback define inline (same as function above) to ensure code self-contained
        def plot_bigcell_heatmap_metricsY(data, metric_labels, class_labels, out_path,
                                          cell_size=CELL_SIZE, annot=True, annot_fontsize=ANNOT_FONTSIZE,
                                          label_fontsize=LABEL_FONTSIZE, cmap="YlGnBu", fmt=".3f", dpi=DPI,
                                          title="Per-class metrics (metrics on Y-axis)"):
            import numpy as _np
            import matplotlib.pyplot as _plt
            import seaborn as _sns

            data = _np.asarray(data)
            n_rows, n_cols = data.shape
            fig_w = max(6, n_cols * cell_size)
            fig_h = max(3, n_rows * cell_size)
            fig, ax = _plt.subplots(figsize=(fig_w, fig_h))
            _sns.set(style="white")
            _sns.heatmap(data,
                         annot=annot,
                         fmt=fmt,
                         cmap=cmap,
                         xticklabels=class_labels,
                         yticklabels=metric_labels,
                         linewidths=0.6,
                         linecolor="gray",
                         annot_kws={"size": annot_fontsize, "weight": "bold"},
                         cbar=True,
                         ax=ax)
            ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=label_fontsize)
            ax.set_yticklabels(metric_labels, rotation=0, fontsize=label_fontsize)
            ax.set_title(title, fontsize=label_fontsize + 2)
            try:
                cbar = ax.collections[0].colorbar
                if cbar is not None:
                    cbar.ax.tick_params(labelsize=max(8, label_fontsize - 1))
            except Exception:
                pass
            _plt.tight_layout()
            out_dir = os.path.dirname(out_path)
            if out_dir and not os.path.exists(out_dir):
                os.makedirs(out_dir, exist_ok=True)
            fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
            _plt.close(fig)
            print(f"Saved heatmap: {out_path}")

    # Save both PNG and PDF
    plot_bigcell_heatmap_metricsY(matrix, row_labels, class_names, png_path,
                                  cell_size=CELL_SIZE, annot=True, annot_fontsize=ANNOT_FONTSIZE,
                                  label_fontsize=LABEL_FONTSIZE, cmap="YlGnBu", fmt=".3f", dpi=DPI,
                                  title=f"{label} - Per-class Metrics (Precision, Recall, F1)")
    plot_bigcell_heatmap_metricsY(matrix, row_labels, class_names, pdf_path,
                                  cell_size=CELL_SIZE, annot=True, annot_fontsize=ANNOT_FONTSIZE,
                                  label_fontsize=LABEL_FONTSIZE, cmap="YlGnBu", fmt=".3f", dpi=DPI,
                                  title=f"{label} - Per-class Metrics (Precision, Recall, F1)")

    print(f"Finished {label}. Outputs in: {model_out}")
    return True

def main():
    test_gen = build_test_generator()

    for label, path in MODEL_PATHS.items():
        if not os.path.exists(path):
            print(f"[WARN] Model not found: {path} — skipping {label}")
            continue
        try:
            process_model(label, path, test_gen)
        except Exception as e:
            print(f"[ERROR] {label} failed: {e}")

if __name__ == "__main__":
    main()


Found 4602 images belonging to 10 classes.

--- Processing model: ALO_WOA ---
Loaded: hybrid_alo_woa_best_model.keras
Predicting on test set...
  1/144 ━━━━━━━━━━━━━━━━━━━━ 19s 134ms/step

/Users/simarkalsi/Projects/leafClassification/leaf/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step
Saved CSV: results/metrics_heatmaps_metricsY/ALO_WOA/per_class_metrics_ALO_WOA.csv
Saved heatmap: results/metrics_heatmaps_metricsY/ALO_WOA/per_class_metrics_heatmap_ALO_WOA_metricsY.png
Saved heatmap: results/metrics_heatmaps_metricsY/ALO_WOA/per_class_metrics_heatmap_ALO_WOA_metricsY.pdf
Finished ALO_WOA. Outputs in: results/metrics_heatmaps_metricsY/ALO_WOA

--- Processing model: ALO_DA ---
Loaded: hybrid_alo_da_best_model.keras
Predicting on test set...
144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step
Saved CSV: results/metrics_heatmaps_metricsY/ALO_DA/per_class_metrics_ALO_DA.csv
Saved heatmap: results/metrics_heatmaps_metricsY/ALO_DA/per_class_metrics_heatmap_ALO_DA_metricsY.png
Saved heatmap: results/metrics_heatmaps_metricsY/ALO_DA/per_class_metrics_heatmap_ALO_DA_metricsY.pdf
Finished ALO_DA. Outputs in: results/metrics_heatmaps_metricsY/ALO_DA

--- Processing model: ALO_PSO ---
Loaded: hybrid_alo_pso_best_model.keras
Predicting on test set..

In [1]:
# generate_gradcam_for_all_models.py
"""
Generate Grad-CAM visualizations for all hybrid models.

- Loads models listed in MODEL_PATHS (skip missing)
- Builds test generator from augmented_split_dataset/test (rescale=1./255, shuffle=False)
- For each model: predict on test set, select top-K correct and top-K misclassified by confidence
- Compute Grad-CAM (last conv layer auto-detected) and save overlay PNGs

Run: python generate_gradcam_for_all_models.py
"""

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix

# ----------------- USER CONFIG -----------------
MODEL_PATHS = {
    "ALO_WOA": "hybrid_alo_woa_best_model.keras",
    "ALO_DA" : "hybrid_alo_da_best_model.keras",
    "ALO_PSO": "hybrid_alo_pso_best_model.keras",
    "PSO_WOA": "hybrid_pso_woa_best_model.keras",  # optional
}

TEST_DIR = "augmented_split_dataset/test"
IMG_SIZE = (256, 256)     # must match training input size
BATCH_SIZE = 32
RESCALE = True            # whether generator rescales images by 1./255

OUT_ROOT = "results/gradcam"
os.makedirs(OUT_ROOT, exist_ok=True)

# How many Grad-CAMs to produce per category per model
K_CORRECT = 5
K_MIS = 5

# Grad-CAM parameters
ALPHA = 0.4   # overlay strength (0-1), higher = stronger heatmap overlay
COLORMAP = "jet"
# -------------------------------------------------

def build_test_generator():
    datagen = ImageDataGenerator(rescale=1./255) if RESCALE else ImageDataGenerator()
    gen = datagen.flow_from_directory(
        directory=TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    return gen

def find_last_conv_layer(model):
    # heuristic: find the last layer that is a Conv2D (or DepthwiseConv2D)
    last_conv = None
    for layer in reversed(model.layers):
        from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, SeparableConv2D
        if isinstance(layer, (Conv2D, DepthwiseConv2D, SeparableConv2D)):
            last_conv = layer.name
            break
    if last_conv is None:
        # fallback: look for layer with 4D output
        for layer in reversed(model.layers):
            if len(layer.output_shape) == 4:
                last_conv = layer.name
                break
    return last_conv

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    img_array: preprocessed image with shape (1, H, W, C) matching model input
    model: loaded keras model
    last_conv_layer_name: string name of conv layer to target
    pred_index: index of class to compute Grad-CAM for (if None use model.predict argmax)
    returns heatmap (H, W) with values in [0,1]
    """
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # compute gradients of top predicted class wrt conv outputs
    grads = tape.gradient(class_channel, conv_outputs)

    # guided gradients / global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]  # (H_conv, W_conv, C)
    pooled_grads = pooled_grads.numpy()
    conv_outputs = conv_outputs.numpy()

    # weigh conv outputs by pooled grads
    for i in range(pooled_grads.shape[-1]):
        conv_outputs[:, :, i] *= pooled_grads[i]

    heatmap = np.mean(conv_outputs, axis=-1)
    # relu and normalize
    heatmap = np.maximum(heatmap, 0)
    max_val = np.max(heatmap) if np.max(heatmap) != 0 else 1e-9
    heatmap /= max_val
    heatmap = np.clip(heatmap, 0, 1)
    # resize to original image size
    heatmap = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE[0], IMG_SIZE[1])).numpy().squeeze()
    return heatmap

def overlay_heatmap_on_image(orig_img, heatmap, alpha=ALPHA, cmap=COLORMAP):
    """
    orig_img: original image as array (H,W,3) in range [0,255] or [0,1]
    heatmap: 2D array (H,W) with values in [0,1]
    returns overlayed image (uint8)
    """
    import matplotlib.cm as cm
    if orig_img.dtype != np.uint8:
        # scale to 0-255
        img_uint8 = np.clip(orig_img * 255.0, 0, 255).astype(np.uint8)
    else:
        img_uint8 = orig_img

    # get color map
    colormap = cm.get_cmap(cmap)
    heatmap_colored = colormap(heatmap)[:, :, :3]  # (H,W,3) float in [0,1]

    overlay = heatmap_colored * (alpha * 255.0) + img_uint8 * (1 - alpha)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay

def save_gradient_visualizations_for_model(label, model_path, test_gen, k_correct=K_CORRECT, k_mis=K_MIS):
    out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n==== Processing model {label} ====")
    model = load_model(model_path)
    print("Model loaded.")

    last_conv = find_last_conv_layer(model)
    if last_conv is None:
        print("[WARN] Could not find a convolutional layer in model. Grad-CAM may fail.")
    else:
        print("Using last conv layer:", last_conv)

    # predict on test set (batchwise)
    y_prob = model.predict(test_gen, verbose=1)
    if y_prob.ndim == 1 or (y_prob.ndim == 2 and y_prob.shape[1] == 1):
        p = y_prob.ravel()
        y_prob = np.vstack([1-p, p]).T
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    # compute confidences (probability for predicted class)
    pred_confidences = y_prob[np.arange(len(y_pred)), y_pred]

    # identify correct & misclassified indices
    correct_idx = np.where(y_pred == y_true)[0]
    mis_idx = np.where(y_pred != y_true)[0]

    # sort correct by descending confidence -> pick top k_correct
    correct_sorted = correct_idx[np.argsort(-pred_confidences[correct_idx])]
    mis_sorted = mis_idx[np.argsort(-pred_confidences[mis_idx])]

    sel_correct = correct_sorted[:k_correct]
    sel_mis = mis_sorted[:k_mis]

    # helper to load raw image from filepath (test_gen.filepaths)
    filepaths = getattr(test_gen, "filepaths", None)
    if filepaths is None:
        # try generator.filenames + directory
        filenames = getattr(test_gen, "filenames", None)
        base_dir = getattr(test_gen, "directory", None)
        if filenames is not None and base_dir is not None:
            filepaths = [os.path.join(base_dir, fn) for fn in filenames]
        else:
            raise RuntimeError("test_gen must expose filepaths or filenames + directory")

    def load_original_image(fp):
        # returns array in [0,1]
        img = load_img(fp, target_size=IMG_SIZE)
        arr = img_to_array(img) / 255.0
        return arr

    # function to preprocess for model (if generator used rescale=1./255, we do same)
    def preprocess_for_model(img_arr):
        # img_arr is in [0,1], model expects rescaled likewise
        return np.expand_dims(img_arr, axis=0).astype(np.float32)

    # save examples
    def save_example(idx, category):
        fp = filepaths[idx]
        orig = load_original_image(fp)  # [0,1]
        inp = preprocess_for_model(orig)  # shape (1,H,W,3)
        pred_class = int(y_pred[idx])
        pred_name = class_names[pred_class]
        true_class = int(y_true[idx])
        true_name = class_names[true_class]
        conf = float(pred_confidences[idx])

        # grad-cam heatmap for predicted class
        try:
            heatmap = make_gradcam_heatmap(inp, model, last_conv, pred_index=pred_class)
        except Exception as e:
            print(f"[WARN] Grad-CAM failed for idx {idx}: {e}")
            return

        overlay = overlay_heatmap_on_image(orig, heatmap, alpha=ALPHA, cmap=COLORMAP)
        # also prepare a side-by-side figure: original | heatmap | overlay
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(orig)
        axes[0].axis("off")
        axes[0].set_title(f"Original\nTrue:{true_name}")

        axes[1].imshow(heatmap, cmap=COLORMAP)
        axes[1].axis("off")
        axes[1].set_title(f"Grad-CAM\nPred:{pred_name}\nConf:{conf:.3f}")

        axes[2].imshow(overlay)
        axes[2].axis("off")
        axes[2].set_title("Overlay")

        out_fp = os.path.join(out_dir, category, f"idx{idx}_true_{true_name}_pred_{pred_name}_conf_{conf:.3f}.png")
        os.makedirs(os.path.dirname(out_fp), exist_ok=True)
        plt.tight_layout()
        fig.savefig(out_fp, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print("Saved:", out_fp)

    # save correct examples
    print(f"Saving up to {len(sel_correct)} high-confidence correct examples...")
    for i in sel_correct:
        save_example(int(i), "correct")

    # save misclassified examples
    print(f"Saving up to {len(sel_mis)} high-confidence misclassified examples...")
    for i in sel_mis:
        save_example(int(i), "misclassified")

    # OPTIONAL: save a small CSV summary of picked indices
    import csv
    summary_csv = os.path.join(out_dir, "picked_examples_summary.csv")
    with open(summary_csv, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["index","filepath","true_label","pred_label","pred_name","true_name","confidence","category"])
        for i in sel_correct:
            fp = filepaths[i]; t=y_true[i]; p=y_pred[i]; c=pred_confidences[i]
            writer.writerow([i, fp, int(t), int(p), class_names[int(p)], class_names[int(t)], float(c), "correct"])
        for i in sel_mis:
            fp = filepaths[i]; t=y_true[i]; p=y_pred[i]; c=pred_confidences[i]
            writer.writerow([i, fp, int(t), int(p), class_names[int(p)], class_names[int(t)], float(c), "misclassified"])
    print("Saved summary CSV:", summary_csv)

def main():
    test_gen = build_test_generator()
    for label, path in MODEL_PATHS.items():
        if not os.path.exists(path):
            print(f"[WARN] Model not found: {path} — skipping {label}")
            continue
        try:
            save_gradient_visualizations_for_model(label, path, test_gen, k_correct=K_CORRECT, k_mis=K_MIS)
        except Exception as e:
            print(f"[ERROR] {label} failed: {e}")

if __name__ == "__main__":
    main()


Found 4602 images belonging to 10 classes.

==== Processing model ALO_WOA ====


2025-09-14 16:00:47.322917: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2025-09-14 16:00:47.322968: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-09-14 16:00:47.322974: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-09-14 16:00:47.323185: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-14 16:00:47.323209: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model loaded.
Using last conv layer: conv2d_137


/Users/simarkalsi/Projects/leafClassification/leaf/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2025-09-14 16:00:47.795476: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step
Saving up to 5 high-confidence correct examples...
[WARN] Grad-CAM failed for idx 3257: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 3659: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 3664: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 3671: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 3673: The layer sequential_45 has never been called and thus has no defined output.
Saving up to 5 high-confidence misclassified examples...
[WARN] Grad-CAM failed for idx 1117: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 300: The layer sequential_45 has never been called and thus has no defined output.
[WARN] Grad-CAM failed for idx 3129: The layer